<a href="https://colab.research.google.com/github/tomasrojas88/Data_Science_1/blob/main/TP_Final_NLP_DeepLearning_Tomas_Rojas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Importo pandas para trabajar con el dataset
import pandas as pd

# Defino la URL del archivo Excel subido a GitHub
url = "https://github.com/tomasrojas88/Data_Science_1/raw/main/Base%20de%20datos%20CURSO%20NPS.xlsx"

# Cargo el archivo Excel desde GitHub
df = pd.read_excel(url)

# Muestro las primeras filas para verificar que la carga sea correcta
df.head()

,ID,Puntaje,Comentario
0,1,10,"Porque cuando tuve un accidente ,me solucionar..."
1,2,7,NaN
2,3,10,"Porque desde que la uso no tuve problemas, ade..."
3,4,9,Porque la aseguradora tiene lo que necesito y ...
4,5,8,Porque AFORTUNADAMENTE no tuve un siniestro y ...


In [2]:
# Reviso la cantidad de filas y columnas del dataset
print("Dimensiones del dataset:", df.shape)

# Veo los nombres de las columnas para confirmar su estructura
print("Columnas:", df.columns.tolist())

# Verifico tipos de datos y valores nulos
df.info()

# Cuento los valores faltantes por columna
print("\nValores nulos por columna:")
print(df.isnull().sum())

# Reviso cómo se distribuyen los puntajes
print("\nDistribución de la variable Puntaje:")
print(df["Puntaje"].value_counts().sort_index())

Dimensiones del dataset: (5645, 3)
Columnas: ['ID', 'Puntaje', 'Comentario']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5645 entries, 0 to 5644
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   ID          5645 non-null   int64 
 1   Puntaje     5645 non-null   int64 
 2   Comentario  4008 non-null   object
dtypes: int64(2), object(1)
memory usage: 132.4+ KB

Valores nulos por columna:
ID               0
Puntaje          0
Comentario    1637
dtype: int64

Distribución de la variable Puntaje:
Puntaje
0       61
1       17
2       16
3       26
4       46
5      218
6      123
7      353
8      748
9      763
10    3274
Name: count, dtype: int64


In [3]:
# Elimino los registros sin comentario porque no sirven para el análisis de texto
df_nlp = df.dropna(subset=["Comentario"]).copy()

# Verifico cuántos registros quedan después de quitar los comentarios nulos
print("Dimensiones del dataset para NLP:", df_nlp.shape)

# Reviso nuevamente los valores nulos
print("\nValores nulos en la nueva base:")
print(df_nlp.isnull().sum())

# Muestro las primeras filas de la nueva base
df_nlp.head()

Dimensiones del dataset para NLP: (4008, 3)

Valores nulos en la nueva base:
ID            0
Puntaje       0
Comentario    0
dtype: int64


,ID,Puntaje,Comentario
0,1,10,"Porque cuando tuve un accidente ,me solucionar..."
2,3,10,"Porque desde que la uso no tuve problemas, ade..."
3,4,9,Porque la aseguradora tiene lo que necesito y ...
4,5,8,Porque AFORTUNADAMENTE no tuve un siniestro y ...
5,6,8,por satisfaccion


In [4]:
# Creo una función para transformar el puntaje NPS en una variable binaria
def clasificar_sentimiento_nps(puntaje):
    # Considero positivos a los promotores (9 y 10)
    if puntaje >= 9:
        return 1
    # Considero negativos a los detractores (0 a 6)
    elif puntaje <= 6:
        return 0
    # Dejo como nulos a los pasivos (7 y 8) para excluirlos del modelo
    else:
        return None

# Aplico la transformación sobre la columna Puntaje
df_nlp["target"] = df_nlp["Puntaje"].apply(clasificar_sentimiento_nps)

# Reviso la distribución inicial de la nueva variable objetivo
print("Distribución inicial de target:")
print(df_nlp["target"].value_counts(dropna=False))

Distribución inicial de target:
target
1.0    2979
NaN     675
0.0     354
Name: count, dtype: int64


In [5]:
# Elimino los casos pasivos porque no los voy a usar en la clasificación binaria
df_modelo = df_nlp.dropna(subset=["target"]).copy()

# Convierto la variable objetivo a entero
df_modelo["target"] = df_modelo["target"].astype(int)

# Verifico cuántos registros quedan para modelar
print("Dimensiones de la base para modelado:", df_modelo.shape)

# Reviso la distribución final de la variable objetivo
print("\nDistribución final de target:")
print(df_modelo["target"].value_counts())

# Muestro algunas filas para controlar el resultado
df_modelo[["Puntaje", "Comentario", "target"]].head()

Dimensiones de la base para modelado: (3333, 4)

Distribución final de target:
target
1    2979
0     354
Name: count, dtype: int64


,Puntaje,Comentario,target
0,10,"Porque cuando tuve un accidente ,me solucionar...",1
2,10,"Porque desde que la uso no tuve problemas, ade...",1
3,9,Porque la aseguradora tiene lo que necesito y ...,1
6,9,"Por el asesor de seguros de mi zona, me gusta ...",1
8,10,"Buen servicio, aunque aún no he necesitado que...",1


In [7]:
# Importo la librería re para hacer limpieza básica de texto
import re

# Creo una función para limpiar los comentarios y dejar solo texto útil
def limpiar_texto(texto):
    # Asegurarse de que el input sea un string
    texto = str(texto)
    # Convierto el texto a minúsculas para unificar palabras
    texto = texto.lower()

    # Elimino tildes simples reemplazando vocales acentuadas
    texto = texto.replace("á", "a").replace("é", "e").replace("í", "i").replace("ó", "o").replace("ú", "u")

    # Elimino signos de puntuación y caracteres especiales
    texto = re.sub(r"[^a-zA-ZñÑ\s]", " ", texto)

    # Elimino espacios repetidos
    texto = re.sub(r"\s+", " ", texto).strip()

    return texto

# Aplico la limpieza sobre la columna de comentarios
df_modelo["comentario_limpio"] = df_modelo["Comentario"].apply(limpiar_texto)

# Muestro ejemplos para comparar comentario original y comentario limpio
df_modelo[["Comentario", "comentario_limpio"]].head(10)

,Comentario,comentario_limpio
0,"Porque cuando tuve un accidente ,me solucionar...",porque cuando tuve un accidente me solucionaro...
2,"Porque desde que la uso no tuve problemas, ade...",porque desde que la uso no tuve problemas adem...
3,Porque la aseguradora tiene lo que necesito y ...,porque la aseguradora tiene lo que necesito y ...
6,"Por el asesor de seguros de mi zona, me gusta ...",por el asesor de seguros de mi zona me gusta l...
8,"Buen servicio, aunque aún no he necesitado que...",buen servicio aunque aun no he necesitado que ...
9,Asé 70 años que tengo seguro con ustedes,ase años que tengo seguro con ustedes
10,"Estoy satisfecho con el servicio, pero me resu...",estoy satisfecho con el servicio pero me resul...
11,Es muy recomendable,es muy recomendable
12,"Muy satisfecho por la promotora, que siempre e...",muy satisfecho por la promotora que siempre es...
13,Porque es lo que me parece,porque es lo que me parece


In [8]:
# Importo nltk y descargo las stopwords en español
import nltk
from nltk.corpus import stopwords

# Descargo el recurso de stopwords para poder usarlo en el análisis
nltk.download("stopwords")

# Guardo las stopwords en español en una lista para filtrar palabras sin contenido semántico fuerte
stop_words = set(stopwords.words("spanish"))

# Creo una función para eliminar stopwords del texto ya limpio
def eliminar_stopwords(texto):
    # Separo el texto en palabras
    palabras = texto.split()

    # Conservo solo las palabras que no están en la lista de stopwords
    palabras_filtradas = [palabra for palabra in palabras if palabra not in stop_words]

    # Vuelvo a unir las palabras en un solo texto
    return " ".join(palabras_filtradas)

# Aplico la eliminación de stopwords sobre la columna de texto limpio
df_modelo["comentario_sin_stopwords"] = df_modelo["comentario_limpio"].apply(eliminar_stopwords)

# Comparo el texto limpio con la nueva versión sin stopwords
df_modelo[["comentario_limpio", "comentario_sin_stopwords"]].head(10)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


,comentario_limpio,comentario_sin_stopwords
0,porque cuando tuve un accidente me solucionaro...,accidente solucionaron rapidamente
2,porque desde que la uso no tuve problemas adem...,uso problemas ademas facil gestionar
3,porque la aseguradora tiene lo que necesito y ...,aseguradora necesito mas hno gracias tambien a...
6,por el asesor de seguros de mi zona me gusta l...,asesor seguros zona gusta atencion personaliza...
8,buen servicio aunque aun no he necesitado que ...,buen servicio aunque aun necesitado paguen
9,ase años que tengo seguro con ustedes,ase años seguro ustedes
10,estoy satisfecho con el servicio pero me resul...,satisfecho servicio resulta dificil costear cu...
11,es muy recomendable,recomendable
12,muy satisfecho por la promotora que siempre es...,satisfecho promotora siempre atenta posibles p...
13,porque es lo que me parece,parece
